# Batch Prediction for Test Set



## Enter the MMSegmentation main directory

In [1]:
import os
os.chdir('mmsegmentation')

In [2]:
os.getcwd()

'/home/featurize/work/Mask2Former/mmsegmentation'

## Import Packages

In [3]:
import os
import numpy as np
import cv2
from tqdm import tqdm

from mmseg.apis import init_model, inference_model, show_result_pyplot
import mmcv

import matplotlib.pyplot as plt
%matplotlib inline

/environment/miniconda3/envs/py38/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load Model

In [4]:
# 模型 config 配置文件
config_file = '../WeightandCode/trans-xylem/ZihaoDataset_Mask2Former_2025.03.07.py'

checkpoint_file = '../WeightandCode/trans-xylem/best_mIoU_iter_11000.pth'

# 计算硬件
# device = 'cpu'
device = 'cuda:0'

In [5]:
model = init_model(config_file, checkpoint_file, device=device)

Loads checkpoint by local backend from path: ../WeightandCode/trans-xylem/best_mIoU_iter_11000.pth


## Specify color schemes for each category

In [6]:
# BGR
palette = [
    ['background', [0,0,0]],
    ['Xylem', [255,255,255]]         
]

palette_dict = {}
for idx, each in enumerate(palette):
    palette_dict[idx] = each[1]

In [7]:
palette_dict

{0: [0, 0, 0], 1: [255, 255, 255]}

## Specify the test set path (can also be changed to the folder path of images to be tested)

In [8]:
PATH_IMAGE = '../Input'

In [9]:
os.chdir(PATH_IMAGE)

## Single Image Prediction Function

In [10]:
opacity = 0

In [11]:
def process_single_img(img_path, save=False):
    
    img_bgr = cv2.imread(img_path)

    result = inference_model(model, img_bgr)
    pred_mask = result.pred_sem_seg.data[0].cpu().numpy()
    pred_mask_bgr = np.zeros((pred_mask.shape[0], pred_mask.shape[1], 3))
    for idx in palette_dict.keys():
        pred_mask_bgr[np.where(pred_mask==idx)] = palette_dict[idx]
    pred_mask_bgr = pred_mask_bgr.astype('uint8')
    pred_viz = cv2.addWeighted(img_bgr, opacity, pred_mask_bgr, 1-opacity, 0)
    
    if save:
        save_path = os.path.join('../Output/'+img_path.split('/')[-1])
        cv2.imwrite(save_path, pred_viz)


## Batch Prediction for Test Set

In [12]:
for each in tqdm(os.listdir()):
    process_single_img(each, save=True)

  0%|          | 0/32 [00:00<?, ?it/s]/environment/miniconda3/envs/py38/lib/python3.8/site-packages/mmdet/models/layers/positional_encoding.py:103: UserWarning: __floordiv__ is deprecated, and its behavior will change in a future version of pytorch. It currently rounds toward 0 (like the 'trunc' function NOT 'floor'). This results in incorrect rounding for negative values. To keep the current behavior, use torch.div(a, b, rounding_mode='trunc'), or for actual floor division, use torch.div(a, b, rounding_mode='floor').
  dim_t = self.temperature**(2 * (dim_t // 2) / self.num_feats)
/environment/miniconda3/envs/py38/lib/python3.8/site-packages/torch/functional.py:445: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at  ../aten/src/ATen/native/TensorShape.cpp:2157.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
100%|██████████| 32/32 [00:45<00:00,  1.43s/it]


The prediction results are saved in the `./Output` directory.

## Delete unnecessary files automatically generated by the system

### View redundant files to be deleted

In [13]:
!find . -iname '__MACOSX'

In [14]:
!find . -iname '.DS_Store'

In [15]:
!find . -iname '.ipynb_checkpoints'

### Delete unnecessary files

In [16]:
!for i in `find . -iname '__MACOSX'`; do rm -rf $i;done

In [17]:
!for i in `find . -iname '.DS_Store'`; do rm -rf $i;done

In [18]:
!for i in `find . -iname '.ipynb_checkpoints'`; do rm -rf $i;done

### Verify that redundant files have been deleted

In [19]:
!find . -iname '__MACOSX'

In [20]:
!find . -iname '.DS_Store'

In [21]:
!find . -iname '.ipynb_checkpoints'

### Calculate the area under a 10X magnification scope

In [22]:
import os
import re
import numpy as np
import pandas as pd
from PIL import Image

input_folder = r'../Output'                         
output_path  = r'../Output/10xXylemArea.xlsx'       
target_color = (255, 255, 255)                     
tolerance    = 2                                   
microns_per_pixel = 100 / 293                       
recursive    = False                               
# ---------------------------------

IMG_EXTS = ('.png', '.jpg', '.jpeg', '.tif', '.tiff', '.bmp')

def _count_target_pixels_from_array(rgb_array: np.ndarray, target_rgb, tol: int) -> int:
    arr = rgb_array.astype(np.int16)
    tgt = np.array(target_rgb, dtype=np.int16).reshape(1, 1, 3)
    diff = np.abs(arr - tgt)
    mask = (diff <= tol).all(axis=2)
    return int(mask.sum())

def count_target_pixels(image_path, target_rgb, tol=0) -> int:
    with Image.open(image_path) as im:
        im = im.convert('RGB')  # 忽略透明度
        arr = np.array(im, dtype=np.uint8)
    return _count_target_pixels_from_array(arr, target_rgb, tol)

def calculate_percentage(part: int, total: int) -> float:
    if total <= 0:
        return 0.0
    return part * 100.0 / total

def _iter_images(folder: str, recursive: bool):
    if recursive:
        for root, _, files in os.walk(folder):
            for f in files:
                if f.lower().endswith(IMG_EXTS):
                    yield os.path.join(root, f)
    else:
        for f in os.listdir(folder):
            p = os.path.join(folder, f)
            if os.path.isfile(p) and f.lower().endswith(IMG_EXTS):
                yield p

def save_table(df: pd.DataFrame, out_path: str):
    """
    优先尝试写 Excel；若缺少 openpyxl/xlsxwriter，则自动退化为 CSV。
    """
    ext = os.path.splitext(out_path)[1].lower()
    if ext in ('.xlsx', '.xls'):
        try:
            # 让 pandas 自选引擎；若没装会抛 ModuleNotFoundError
            df.to_excel(out_path, index=False)
            print(f"[保存] Excel -> {out_path}")
            return
        except ModuleNotFoundError as e:
            print(f"[提示] Excel 引擎缺失({e}), 自动改存 CSV。")
            out_path = re.sub(r'\.xlsx?$', '.csv', out_path, flags=re.IGNORECASE)

    # CSV 路径（用户原本就给了 .csv 或 Excel 失败时的回退）
    df.to_csv(out_path, index=False, encoding='utf-8-sig')
    print(f"[保存] CSV -> {out_path}")

def process_folder_percentage(
    input_folder: str,
    output_path: str,
    target_color=(255, 255, 255),
    tolerance: int = 0,
    microns_per_pixel: float = 100/293,
    recursive: bool = False
):
    """
    逐图统计目标颜色像素数、占比，并将目标像素面积换算为 μm²。
    microns_per_pixel: 每像素对应的微米数（例如 100μm/293px -> 100/293）。
    """
    if not os.path.isdir(input_folder):
        raise NotADirectoryError(f"输入文件夹不存在：{input_folder}")

    pixel_area_um2 = float(microns_per_pixel) ** 2  # 每像素面积 (μm²)

    rows = []
    total_images = 0
    processed = 0

    for img_path in _iter_images(input_folder, recursive):
        total_images += 1
        try:
            with Image.open(img_path) as im:
                im = im.convert('RGB')
                w, h = im.size
                arr = np.array(im, dtype=np.uint8)

            total_pixels = w * h
            target_pixels = _count_target_pixels_from_array(arr, target_color, tolerance)
            percentage = calculate_percentage(target_pixels, total_pixels)
            real_area_um2 = target_pixels * pixel_area_um2

            rows.append({
                'Image Name': os.path.basename(img_path),
                'Width (px)': w,
                'Height (px)': h,
                'Total Pixels': total_pixels,
                'Target Pixels': target_pixels,
                'Target Area (μm²)': real_area_um2,
                'Percentage (%)': percentage,
                'Target Color (RGB)': str(tuple(target_color)),
                'Tolerance': tolerance,
                'Microns per Pixel (μm/px)': microns_per_pixel,
            })
            processed += 1
        except Exception as e:
            print(f"[跳过] {img_path}: {e}")

    df = pd.DataFrame(rows)
    # 美化数值
    if not df.empty:
        for col in ['Percentage (%)']:
            df[col] = df[col].astype(float).round(6)
        for col in ['Target Area (μm²)']:
            df[col] = df[col].astype(float).round(3)

    save_table(df, output_path)

    print("\n========== 汇总 ==========")
    print(f"发现图片：{total_images}")
    print(f"成功统计：{processed}")
    print(f"输出文件：{output_path if output_path.lower().endswith(('.xlsx','.xls','.csv')) else '（已按规则保存）'}")

# ===== 实际执行 =====
process_folder_percentage(
    input_folder=input_folder,
    output_path=output_path,
    target_color=target_color,
    tolerance=tolerance,
    microns_per_pixel=microns_per_pixel,
    recursive=recursive
)


[提示] Excel 引擎缺失(No module named 'openpyxl'), 自动改存 CSV。
[保存] CSV -> ../Output/10xXylemArea.csv

========== 汇总 ==========
发现图片：7
成功统计：7
输出文件：../Output/10xXylemArea.xlsx


### Calculate the area under a 20X magnification scope

In [23]:
# ==== 一键运行（Jupyter 单元格） ====

import os
import re
import numpy as np
import pandas as pd
from PIL import Image

# --------- 可调参数 ----------
input_folder = r'../Output'                         # 输入图片文件夹
output_path  = r'../Output/20xXylemArea.xlsx'       # 输出文件：.xlsx 或 .csv 均可
target_color = (255, 255, 255)                      # 目标颜色(R,G,B)，示例：白色
tolerance    = 2                                    # 容差（0=完全匹配；值大更宽松）
microns_per_pixel = 50 / 293                       # 每像素的微米数（例：100 μm ≈ 293 px）
recursive    = False                                # 是否递归子目录
# ---------------------------------

IMG_EXTS = ('.png', '.jpg', '.jpeg', '.tif', '.tiff', '.bmp')

def _count_target_pixels_from_array(rgb_array: np.ndarray, target_rgb, tol: int) -> int:
    """
    rgb_array: H x W x 3, dtype uint8
    target_rgb: (R,G,B)
    tol: 容差
    """
    # 用 int16 防止减法时 uint8 溢出
    arr = rgb_array.astype(np.int16)
    tgt = np.array(target_rgb, dtype=np.int16).reshape(1, 1, 3)
    diff = np.abs(arr - tgt)
    mask = (diff <= tol).all(axis=2)
    return int(mask.sum())

def count_target_pixels(image_path, target_rgb, tol=0) -> int:
    with Image.open(image_path) as im:
        im = im.convert('RGB')  # 忽略透明度
        arr = np.array(im, dtype=np.uint8)
    return _count_target_pixels_from_array(arr, target_rgb, tol)

def calculate_percentage(part: int, total: int) -> float:
    if total <= 0:
        return 0.0
    return part * 100.0 / total

def _iter_images(folder: str, recursive: bool):
    if recursive:
        for root, _, files in os.walk(folder):
            for f in files:
                if f.lower().endswith(IMG_EXTS):
                    yield os.path.join(root, f)
    else:
        for f in os.listdir(folder):
            p = os.path.join(folder, f)
            if os.path.isfile(p) and f.lower().endswith(IMG_EXTS):
                yield p

def save_table(df: pd.DataFrame, out_path: str):
    ext = os.path.splitext(out_path)[1].lower()
    if ext in ('.xlsx', '.xls'):
        try:
            # 让 pandas 自选引擎；若没装会抛 ModuleNotFoundError
            df.to_excel(out_path, index=False)
            print(f"[保存] Excel -> {out_path}")
            return
        except ModuleNotFoundError as e:
            print(f"[提示] Excel 引擎缺失({e}), 自动改存 CSV。")
            out_path = re.sub(r'\.xlsx?$', '.csv', out_path, flags=re.IGNORECASE)

    # CSV 路径（用户原本就给了 .csv 或 Excel 失败时的回退）
    df.to_csv(out_path, index=False, encoding='utf-8-sig')
    print(f"[save] CSV -> {out_path}")

def process_folder_percentage(
    input_folder: str,
    output_path: str,
    target_color=(255, 255, 255),
    tolerance: int = 0,
    microns_per_pixel: float = 100/293,
    recursive: bool = False
):

    if not os.path.isdir(input_folder):
        raise NotADirectoryError(f"输入文件夹不存在：{input_folder}")

    pixel_area_um2 = float(microns_per_pixel) ** 2  # 每像素面积 (μm²)

    rows = []
    total_images = 0
    processed = 0

    for img_path in _iter_images(input_folder, recursive):
        total_images += 1
        try:
            with Image.open(img_path) as im:
                im = im.convert('RGB')
                w, h = im.size
                arr = np.array(im, dtype=np.uint8)

            total_pixels = w * h
            target_pixels = _count_target_pixels_from_array(arr, target_color, tolerance)
            percentage = calculate_percentage(target_pixels, total_pixels)
            real_area_um2 = target_pixels * pixel_area_um2

            rows.append({
                'Image Name': os.path.basename(img_path),
                'Width (px)': w,
                'Height (px)': h,
                'Total Pixels': total_pixels,
                'Target Pixels': target_pixels,
                'Target Area (μm²)': real_area_um2,
                'Percentage (%)': percentage,
                'Target Color (RGB)': str(tuple(target_color)),
                'Tolerance': tolerance,
                'Microns per Pixel (μm/px)': microns_per_pixel,
            })
            processed += 1
        except Exception as e:
            print(f"[跳过] {img_path}: {e}")

    df = pd.DataFrame(rows)
    # 美化数值
    if not df.empty:
        for col in ['Percentage (%)']:
            df[col] = df[col].astype(float).round(6)
        for col in ['Target Area (μm²)']:
            df[col] = df[col].astype(float).round(3)

    save_table(df, output_path)

    print("\n========== 汇总 ==========")
    print(f"发现图片：{total_images}")
    print(f"成功统计：{processed}")
    print(f"输出文件：{output_path if output_path.lower().endswith(('.xlsx','.xls','.csv')) else '（已按规则保存）'}")

# ===== 实际执行 =====
process_folder_percentage(
    input_folder=input_folder,
    output_path=output_path,
    target_color=target_color,
    tolerance=tolerance,
    microns_per_pixel=microns_per_pixel,
    recursive=recursive
)


[提示] Excel 引擎缺失(No module named 'openpyxl'), 自动改存 CSV。
[save] CSV -> ../Output/20xXylemArea.csv

========== 汇总 ==========
发现图片：32
成功统计：32
输出文件：../Output/20xXylemArea.xlsx


### Xylem sectioning

In [24]:
import os
import cv2
import numpy as np
from mmseg.apis import init_model, inference_model
from tqdm import tqdm

config_file = '../WeightandCode/trans-xylem/ZihaoDataset_Mask2Former_2025.03.07.py'
checkpoint_file = '../WeightandCode/trans-xylem/best_mIoU_iter_11000.pth'

model = init_model(config_file, checkpoint_file, device='cuda:0')

input_folder = '../Input/'  
output_folder = '../Output/'  
os.makedirs(output_folder, exist_ok=True)

def apply_mask_to_image(image, mask):
    mask = np.squeeze(mask)
    if mask.shape[:2] != image.shape[:2]:
        original_shape = image.shape[:2]
        mask = cv2.resize(mask, (original_shape[1], original_shape[0]), interpolation=cv2.INTER_NEAREST)
    
    masked_image = image.copy()
    masked_image[mask == 0] = [0, 0, 0]  
    return masked_image

for img_filename in tqdm(os.listdir(input_folder)):
    img_path = os.path.join(input_folder, img_filename)

    if not img_filename.lower().endswith((".png", ".jpg", ".jpeg", ".tif", ".tiff")):
        continue

    image = cv2.imread(img_path)
    if image is None:
        print(f"Failed to read {img_path}")
        continue

    result = inference_model(model, img_path)

    mask = result.pred_sem_seg.data.cpu().numpy().astype(np.uint8)

    masked_image = apply_mask_to_image(image, mask)

    output_path = os.path.join(output_folder, img_filename)
    cv2.imwrite(output_path, masked_image)

print(f"finish: {output_folder}")


Loads checkpoint by local backend from path: ../WeightandCode/trans-xylem/best_mIoU_iter_11000.pth


  0%|          | 0/32 [00:00<?, ?it/s]/environment/miniconda3/envs/py38/lib/python3.8/site-packages/mmdet/models/layers/positional_encoding.py:103: UserWarning: __floordiv__ is deprecated, and its behavior will change in a future version of pytorch. It currently rounds toward 0 (like the 'trunc' function NOT 'floor'). This results in incorrect rounding for negative values. To keep the current behavior, use torch.div(a, b, rounding_mode='trunc'), or for actual floor division, use torch.div(a, b, rounding_mode='floor').
  dim_t = self.temperature**(2 * (dim_t // 2) / self.num_feats)
100%|██████████| 32/32 [00:42<00:00,  1.33s/it]

finish: ../Output/
